## 1) installation

In [1]:
!pip install -q "trl==0.29.1" peft
!pip uninstall -y -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 3.0 MB/s eta 0:00:0000:0100:01


## 2) Reglagle GPU

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!rm -rf llm_pii_anonymizer_fr
!git clone https://github.com/ellakonan41/llm_pii_anonymizer_fr.git
import sys
sys.path.append("llm_pii_anonymizer_fr")

from src.annotation import charger_annotations
from src.generation import anonymiser
from src.evaluation import evaluer, evaluer_exemple

Cloning into 'llm_pii_anonymizer_fr'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 84 (delta 39), reused 67 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 166.60 KiB | 2.14 MiB/s, done.
Resolving deltas: 100% (39/39), done.


## 3)charger le test réaliste et faire le contrôle de cohérence

In [3]:
test_reel = charger_annotations("llm_pii_anonymizer_fr/data/test_realiste.jsonl")
textes = [ex["source_text"] for ex in test_reel]
print(len(test_reel), "textes")

def afficher(resultats):
    for cle, valeur in resultats.items():
        if not isinstance(valeur, dict):
            print(f"{cle:<25} {valeur:.1%}")

afficher(evaluer(test_reel, [ex["masked_text"] for ex in test_reel]))

50 textes
protection                100.0%
textes_sans_fuite         100.0%
preservation              100.0%
correspondance_exacte     100.0%


## 4)évaluer la v0 (modèle de base)

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NOM_MODELE = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NOM_MODELE)
tok.padding_side = "left"

model_v0 = AutoModelForCausalLM.from_pretrained(NOM_MODELE, dtype=torch.float16, device_map="cuda")
model_v0.eval()

sorties_v0 = anonymiser(model_v0, tok, textes)
resultats_v0 = evaluer(test_reel, sorties_v0)
afficher(resultats_v0)

del model_v0
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


protection                25.0%
textes_sans_fuite         24.0%
preservation              38.3%
correspondance_exacte     4.0%


## 5)recharger la v1 (modèle de base + adaptateur LoRA)

In [5]:
import glob
from peft import PeftModel

chemin_lora = os.path.dirname(glob.glob("/kaggle/input/**/lora_v1/adapter_config.json", recursive=True)[0])
print("Adaptateur :", chemin_lora)

base = AutoModelForCausalLM.from_pretrained(NOM_MODELE, dtype=torch.float32, device_map="cuda")
model_v1 = PeftModel.from_pretrained(base, chemin_lora).merge_and_unload().to(torch.float16)
model_v1.eval()

Adaptateur : /kaggle/input/notebooks/konanella/04-sft-v1/lora_v1


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 6)évaluer la v1 et comparer

In [6]:
sorties_v1 = anonymiser(model_v1, tok, textes)
resultats_v1 = evaluer(test_reel, sorties_v1)

print(f"{'métrique':<25} {'v0':>8} {'v1':>8}")
for cle in ["protection", "textes_sans_fuite", "preservation", "correspondance_exacte"]:
    print(f"{cle:<25} {resultats_v0[cle]:>8.1%} {resultats_v1[cle]:>8.1%}")

print(f"\n{'type':<20} {'n':>4} {'v0':>8} {'v1':>8}")
for label, n in resultats_v1["effectifs_par_type"].items():
    print(f"{label:<20} {n:>4} {resultats_v0['protection_par_type'][label]:>8.1%} "
          f"{resultats_v1['protection_par_type'][label]:>8.1%}")

  0%|          | 0/4 [00:00<?, ?it/s]

métrique                        v0       v1
protection                   25.0%    97.0%
textes_sans_fuite            24.0%    88.0%
preservation                 38.3%    99.5%
correspondance_exacte         4.0%    48.0%

type                    n       v0       v1
GIVENNAME              52    11.5%   100.0%
SURNAME                47    25.5%   100.0%
DATE                   24    29.2%    95.8%
TITLE                  21    38.1%   100.0%
CITY                   14    28.6%   100.0%
TIME                   12    16.7%    83.3%
TELEPHONENUM           11    27.3%   100.0%
BUILDINGNUM            10    40.0%   100.0%
STREET                 10    30.0%    60.0%
ZIPCODE                10    40.0%   100.0%
EMAIL                   7    42.9%   100.0%
AGE                     5    20.0%   100.0%
SOCIALNUM               4     0.0%   100.0%
CREDITCARDNUMBER        2    50.0%   100.0%
DRIVERLICENSENUM        2     0.0%   100.0%
IDCARDNUM               1     0.0%   100.0%
SEX                     1   100

## 7)analyser les erreurs de la v1

In [7]:
from src.evaluation import entite_protegee, mots_hors_donnees_personnelles

for ex, s in zip(test_reel, sorties_v1):
    mots = frozenset(mots_hors_donnees_personnelles(ex))
    fuites = [f"{e['label']}={e['value']!r}" for e in ex["privacy_mask"] if not entite_protegee(e["value"], s, mots)]
    r = evaluer_exemple(ex, s)
    if fuites or r["nb_mots_conserves"] < r["nb_mots_attendus"]:
        print(ex["uid"])
        print("  ORIGINAL :", ex["source_text"])
        print("  MODÈLE   :", s)
        if fuites:
            print("  FUITES   :", ", ".join(fuites))
        if r["nb_mots_conserves"] < r["nb_mots_attendus"]:
            print(f"  MOTS PERDUS : {r['nb_mots_attendus'] - r['nb_mots_conserves']}")
        print("-" * 80)

reel-007
  ORIGINAL : Bonjour, je souhaite décaler mon rendez-vous de kiné du mardi 8 avril à 17h15. Je suis joignable au 07 81 22 64 09. Merci, Rose Antoine
  MODÈLE   : Bonjour, je souhaite décaler mon rendez-vous de kiné du mardi 8 avril à 17h15. Je suis joignable au [TELEPHONENUM]. Merci, [GIVENNAME]
  FUITES   : DATE='8 avril', TIME='17h15'
--------------------------------------------------------------------------------
reel-008
  ORIGINAL : Compte rendu : patiente née le 12/09/1958, numéro de sécurité sociale 2 58 09 75 112 034 61, admise aux urgences de l'hôpital Pellegrin le 2 mars pour une fracture du poignet.
  MODÈLE   : Compte rendu : patiente née le [DATE], numéro de sécurité sociale [SOCIALNUM], admise aux urgences de l'hôpital [CITY] le [DATE] pour une fracture du poignet.
  MOTS PERDUS : 1
--------------------------------------------------------------------------------
reel-011
  ORIGINAL : RAPPEL : rdv demain 9h00 avec Mme BRUNET Claire, centre dentaire, 14 avenue Jean

## 8)sauvegarder

In [8]:
import json

with open("/kaggle/working/sorties_reel.jsonl", "w", encoding="utf-8") as f:
    for ex, s0, s1 in zip(test_reel, sorties_v0, sorties_v1):
        f.write(json.dumps({"uid": ex["uid"], "v0": s0, "v1": s1}, ensure_ascii=False) + "\n")
with open("/kaggle/working/resultats_reel.json", "w", encoding="utf-8") as f:
    json.dump({"v0": resultats_v0, "v1": resultats_v1}, f, ensure_ascii=False, indent=2)